In [1]:
# Cell 1
# Mục đích: định nghĩa lại client, mb, check, stats (notebook mới phải định nghĩa lại).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # kênh kết nối, location khớp dataset

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi đọc byte quá sớm lộ ra
    if n is None:
        return None
    return round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng số MB sẽ quét, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, tắt cache, đặt trần byte, in processed và billed, trả DataFrame
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,  # đổi MB sang byte
    )
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()  # result() phải đứng trước khi đọc byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [2]:
# Cell 1b
# Mục đích: xem danh sách cột (tên, kiểu) của view JOBS mà không tốn tiền, bằng dry run.
# SELECT * ở đây an toàn vì dry run không chạy thật; khi chạy thật thì vẫn không dùng SELECT *.
sql = "SELECT * FROM `region-us`.INFORMATION_SCHEMA.JOBS"
cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
job = client.query(sql, job_config=cfg)  # dry run: không chạy, không tính tiền
print(len(job.schema))                   # số cột
for field in job.schema:
    print(field.name, field.field_type)

52
creation_time TIMESTAMP
project_id STRING
project_number INTEGER
user_email STRING
principal_subject STRING
job_id STRING
job_type STRING
statement_type STRING
priority STRING
start_time TIMESTAMP
end_time TIMESTAMP
query STRING
state STRING
reservation_id STRING
reservation_group_path STRING
total_bytes_processed INTEGER
total_slot_ms INTEGER
final_execution_duration_ms INTEGER
total_services_sku_slot_ms INTEGER
error_result RECORD
cache_hit BOOLEAN
destination_table RECORD
referenced_tables RECORD
labels RECORD
timeline RECORD
job_stages RECORD
total_bytes_billed INTEGER
transaction_id STRING
parent_job_id STRING
session_info RECORD
dml_statistics RECORD
total_modified_partitions INTEGER
bi_engine_statistics RECORD
query_info RECORD
transferred_bytes INTEGER
materialized_view_statistics RECORD
edition STRING
job_creation_reason RECORD
continuous_query_info RECORD
continuous BOOLEAN
query_dialect STRING
metadata_cache_statistics RECORD
search_statistics RECORD
vector_search_statist

In [3]:
# Cell 1c
# Mục đích: chứng minh byte phụ thuộc vào CỘT được đọc, không phụ thuộc việc gom nhóm; dùng dry run nên không tốn tiền.
sql_a = """
SELECT product_id, cost
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
sql_b = """
SELECT COUNT(product_id), SUM(cost)
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
sql_c = """
SELECT product_id
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
check(sql_a)  # đọc 2 cột
check(sql_b)  # đọc cùng 2 cột, nhưng trả về 1 dòng
check(sql_c)  # chỉ đọc 1 cột: dự đoán số MB nhỏ hơn

Sẽ quét: 0.69 MB
Sẽ quét: 0.69 MB
Sẽ quét: 0.35 MB


In [4]:
# Cell 1d
# Mục đích: so sánh THỜI GIAN chạy job và thời gian tải kết quả của 3 câu trong Cell 1c (cần chạy Cell 1c trước để có sql_a, sql_b, sql_c).
import time

def run_timed(name, sql, max_mb=100):
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,                       # tắt cache để đo thật
        maximum_bytes_billed=max_mb * 1024 * 1024,   # trần byte (đổi MB sang byte)
    )
    t0 = time.time()
    job = client.query(sql, job_config=cfg)
    job.result()                                     # chờ job xong
    t_wait = time.time() - t0                        # thời gian từ lúc gửi đến lúc xong (có cả độ trễ)
    t1 = time.time()
    df = job.result().to_dataframe()                 # tải kết quả về máy
    t_download = time.time() - t1
    t_job = (job.ended - job.started).total_seconds()  # thời gian job thực sự chạy trong BigQuery
    print(
        f"{name}: dòng={len(df)} | processed={mb(job.total_bytes_processed)} MB | "
        f"billed={mb(job.total_bytes_billed)} MB | job={t_job:.2f}s | "
        f"chờ={t_wait:.2f}s | tải về={t_download:.2f}s | slot_ms={job.slot_millis}"
    )

run_timed("A (product_id, cost)", sql_a)
run_timed("B (COUNT, SUM)", sql_b)
run_timed("C (product_id)", sql_c)

A (product_id, cost): dòng=45529 | processed=0.69 MB | billed=10.0 MB | job=0.26s | chờ=1.28s | tải về=2.82s | slot_ms=20
B (COUNT, SUM): dòng=1 | processed=0.69 MB | billed=10.0 MB | job=0.15s | chờ=1.23s | tải về=2.10s | slot_ms=13
C (product_id): dòng=45529 | processed=0.35 MB | billed=10.0 MB | job=0.26s | chờ=1.49s | tải về=1.95s | slot_ms=10


In [5]:
# Cell 2
# Mục đích: liệt kê các query do dbt gửi lên BigQuery trong 48 giờ qua (docs generate hôm nay, test hôm qua).
sql = """
SELECT
  creation_time,                                   -- giờ UTC (giờ Việt Nam = UTC + 7)
  statement_type,
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed / 1024 / 1024, 2) AS billed_mb,
  SUBSTR(query, 1, 160) AS query_head              -- đầu câu SQL, có chú thích của dbt
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 48 HOUR)
  AND job_type = 'QUERY'
  AND query LIKE '/* {"app": "dbt"%'               -- dbt gắn chú thích này vào đầu mỗi query (mình chưa chắc dạng chính xác)
  AND query NOT LIKE '%INFORMATION_SCHEMA.JOBS%'   -- loại chính câu này
ORDER BY creation_time
"""
check(sql)
df = stats(sql)
print(len(df))  # in số dòng trước khi nhìn bảng: 0 nghĩa là bộ lọc quá hẹp
df

Sẽ quét: 0.12 MB
processed: 0.06 MB | billed: 20.0 MB
17


,creation_time,statement_type,processed_mb,billed_mb,query_head
0,2026-10-13 04:53:01.664000+00:00,SELECT,0.35,10.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
1,2026-10-13 04:53:01.665000+00:00,SELECT,0.57,20.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
2,2026-10-13 04:53:01.669000+00:00,SELECT,0.35,10.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
3,2026-10-13 04:53:01.705000+00:00,SELECT,0.35,10.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
4,2026-10-13 05:03:44.727000+00:00,SELECT,0.22,10.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
5,2026-10-13 05:03:44.731000+00:00,SELECT,0.22,10.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
6,2026-10-13 05:03:44.776000+00:00,SELECT,0.00,0.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
7,2026-10-13 05:28:39.402000+00:00,SELECT,0.00,0.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
8,2026-10-13 05:28:39.458000+00:00,SELECT,0.00,0.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."
9,2026-10-13 05:28:39.498000+00:00,SELECT,0.17,10.0,"/* {""app"": ""dbt"", ""dbt_version"": ""1.12.5"", ""pr..."


In [6]:
# Cell 3
# Mục đích: cộng tổng số query và billed theo ngày để trả lời "dbt test tốn bao nhiêu".
sql = """
SELECT
  DATE(creation_time) AS ngay_utc,
  COUNT(*) AS so_query,
  ROUND(SUM(total_bytes_billed) / 1024 / 1024, 1) AS billed_mb
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 72 HOUR)
  AND job_type = 'QUERY'
  AND query LIKE '/* {"app": "dbt"%'
  AND query NOT LIKE '%INFORMATION_SCHEMA.JOBS%'
GROUP BY ngay_utc
ORDER BY ngay_utc
"""
df = stats(sql)
print(len(df))
df

processed: 0.07 MB | billed: 20.0 MB
2


,ngay_utc,so_query,billed_mb
0,2026-10-13,15,90.0
1,2026-10-14,2,100.0


In [7]:
# Cell 4
# Mục đích: xem từng query của dbt là test nào hay là query catalog của docs, và có lấy từ cache không.
import pandas as pd
pd.set_option("display.max_colwidth", 300)  # cho phép hiện cột chữ dài

sql = r"""
SELECT
  creation_time,
  cache_hit,                                                     -- TRUE nếu kết quả lấy từ cache
  REGEXP_EXTRACT(query, r'"node_id": "([^"]+)"') AS node_id,     -- tên node dbt trong chú thích; NULL nếu dbt không ghi
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed / 1024 / 1024, 2) AS billed_mb,
  SUBSTR(REGEXP_REPLACE(query, r'^/\*.*?\*/', ''), 1, 300) AS sql_head  -- bỏ chú thích, lấy 300 ký tự đầu của SQL
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 48 HOUR)
  AND job_type = 'QUERY'
  AND query LIKE '/* {"app": "dbt"%'
  AND query NOT LIKE '%INFORMATION_SCHEMA.JOBS%'
ORDER BY creation_time
"""
check(sql)
df = stats(sql)
print(len(df))
df

Sẽ quét: 0.12 MB
processed: 0.06 MB | billed: 20.0 MB
17


,creation_time,cache_hit,node_id,processed_mb,billed_mb,sql_head
0,2026-10-13 04:53:01.664000+00:00,False,test.bi_dbt.unique_fct_net_sales_order_item_id.69c2ff268d,0.35,10.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\nwith dbt_test__target as (\n\n select order_item_id as unique_field\n from `bq-learning-510104`.`dbt_dev`.`fct_net_sales`\n ..."
1,2026-10-13 04:53:01.665000+00:00,False,test.bi_dbt.relationships_fct_net_sales_product_id__product_id__ref_dim_products_.f594ea4f41,0.57,20.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\nwith child as (\n select product_id as from_field\n from `bq-learning-510104`.`dbt_dev`.`fct_net_sales`\n where produc..."
2,2026-10-13 04:53:01.669000+00:00,False,test.bi_dbt.not_null_fct_net_sales_order_item_id.c5c9bd5463,0.35,10.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\n\n\nselect order_item_id\nfrom `bq-learning-510104`.`dbt_dev`.`fct_net_sales`\nwhere order_item_id is null\n\n\n\n \n \n ..."
3,2026-10-13 04:53:01.705000+00:00,False,test.bi_dbt.not_null_fct_net_sales_product_id.7886f7c168,0.35,10.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\n\n\nselect product_id\nfrom `bq-learning-510104`.`dbt_dev`.`fct_net_sales`\nwhere product_id is null\n\n\n\n \n \n \n ..."
4,2026-10-13 05:03:44.727000+00:00,False,test.bi_dbt.unique_dim_products_product_id.ef7b2a9985,0.22,10.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\nwith dbt_test__target as (\n\n select product_id as unique_field\n from `bq-learning-510104`.`dbt_dev`.`dim_products`\n wher..."
5,2026-10-13 05:03:44.731000+00:00,False,test.bi_dbt.not_null_dim_products_product_id.c8aba288d1,0.22,10.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\n\n\nselect product_id\nfrom `bq-learning-510104`.`dbt_dev`.`dim_products`\nwhere product_id is null\n\n\n\n \n \n \n ..."
6,2026-10-13 05:03:44.776000+00:00,True,test.bi_dbt.relationships_fct_net_sales_product_id__product_id__ref_dim_products_.f594ea4f41,0.00,0.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\nwith child as (\n select product_id as from_field\n from `bq-learning-510104`.`dbt_dev`.`fct_net_sales`\n where produc..."
7,2026-10-13 05:28:39.402000+00:00,True,test.bi_dbt.relationships_fct_net_sales_product_id__product_id__ref_dim_products_.f594ea4f41,0.00,0.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\nwith child as (\n select product_id as from_field\n from `bq-learning-510104`.`dbt_dev`.`fct_net_sales`\n where produc..."
8,2026-10-13 05:28:39.458000+00:00,True,test.bi_dbt.not_null_dim_products_product_id.c8aba288d1,0.00,0.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\n\n\nselect product_id\nfrom `bq-learning-510104`.`dbt_dev`.`dim_products`\nwhere product_id is null\n\n\n\n \n \n \n ..."
9,2026-10-13 05:28:39.498000+00:00,False,test.bi_dbt.accepted_values_dim_products_department__Women.c54b1dac20,0.17,10.0,"\n\n \n select\n count(*) as failures,\n count(*) != 0 as should_warn,\n count(*) != 0 as should_error\n from (\n \n \n \n \n \n\nwith all_values as (\n\n select\n department as value_field,\n count(*) as n_records\n\n from `bq-learning-..."


In [9]:
# Cell 5
# Mục đích: xem 2 query của dbt docs generate đọc view hệ thống nào và tham chiếu bao nhiêu bảng, để hiểu vì sao processed 40 MB nhưng billed 50 MB.
sql = r"""
SELECT
  creation_time,
  REGEXP_EXTRACT_ALL(query, r'[`\w\-\.]+INFORMATION_SCHEMA\.\w+') AS view_he_thong,  -- các view hệ thống được gọi trong query
  ARRAY_LENGTH(referenced_tables) AS so_bang_tham_chieu,                            -- số bảng BigQuery ghi nhận đã đọc
  ROUND(total_bytes_processed / 1024 / 1024, 2) AS processed_mb,
  ROUND(total_bytes_billed / 1024 / 1024, 2) AS billed_mb
FROM `region-us`.INFORMATION_SCHEMA.JOBS
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 48 HOUR)
  AND job_type = 'QUERY'
  AND query LIKE '/* {"app": "dbt"%'
  AND query NOT LIKE '%INFORMATION_SCHEMA.JOBS%'   -- loại chính câu này
  AND query LIKE '%table_catalog%'                 -- chỉ lấy query catalog của docs
ORDER BY creation_time
"""
check(sql)
df = stats(sql)
print(len(df))
df

Sẽ quét: 0.12 MB
processed: 0.06 MB | billed: 20.0 MB
4


,creation_time,view_he_thong,so_bang_tham_chieu,processed_mb,billed_mb
0,2026-10-14 03:36:11.961000+00:00,"[`bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.MATERIALIZED_VIEWS, `bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.TABLE_OPTIONS, `bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.COLUMNS, `bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.COL...",5,40.0,50.0
1,2026-10-14 03:36:12.006000+00:00,"[`bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.MATERIALIZED_VIEWS, `bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.TABLE_OPTIONS, `bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.COLUMNS, `bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS]",5,40.0,50.0
2,2026-10-14 05:08:26.362000+00:00,"[`bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.MATERIALIZED_VIEWS, `bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.TABLE_OPTIONS, `bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.COLUMNS, `bigquery-public-data`.`thelook_ecommerce`.INFORMATION_SCHEMA.COL...",5,40.0,50.0
3,2026-10-14 05:08:26.712000+00:00,"[`bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.MATERIALIZED_VIEWS, `bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.TABLE_OPTIONS, `bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.COLUMNS, `bq-learning-510104`.`dbt_dev`.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS]",5,40.0,50.0
